# Modelado para detección de fraccionamiento

## Dependencias

In [16]:
import os
import itertools
import pickle as pkl

import polars as pl
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import entropy
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score 

import mlflow
import mlflow.sklearn
import mlflow.pyfunc

## Funciones auxiliares

In [2]:
# Función para calcular la entropía de una lista de montos
def calculate_amount_entropy(amounts) -> float:
    # Convertir Series de Polars a lista si es necesario
    if hasattr(amounts, 'to_list'):
        amounts = amounts.to_list()
    
    if len(amounts) == 0:
        return 0.0
    # Calcular value_counts y luego la entropía. np.unique es más eficiente aquí.
    unique_amounts, counts = np.unique(amounts, return_counts=True)
    probabilities = counts / counts.sum()
    return entropy(probabilities)

In [3]:
# Función para calcular el coeficiente de variación
def calculate_coeff_of_variation(amounts: list[float]) -> float:
    if not amounts:
        return 0.0
    amounts_arr = np.array(amounts)
    std_val = np.std(amounts_arr)
    mean_val = np.mean(amounts_arr)
    return std_val / mean_val if mean_val != 0 else 0.0

In [4]:
# Registrar las funciones Python como UDFs en Polars para uso con `map_batches` o `apply`
calculate_amount_entropy_pl = pl.col("transaction_amount").map_batches(
    lambda s: [calculate_amount_entropy(val) for val in s],
    return_dtype=pl.Float64
).alias("amount_entropy")

calculate_coeff_of_variation_pl = pl.col("transaction_amount").map_batches(
    lambda s: [calculate_coeff_of_variation(val) for val in s],
    return_dtype=pl.Float64
).alias("amount_coeff_of_variation")

In [19]:
BASE_DIR = "../"
DATA_DIR = os.path.join(BASE_DIR, "data")
RAW_DATA_DIR = os.path.join(DATA_DIR, "raw")
FILE_NAME = os.listdir(RAW_DATA_DIR)[0]
FILE_PATH = os.path.join(RAW_DATA_DIR, FILE_NAME)
MODEL_DIR = os.path.join(BASE_DIR, "models")
SCALER_PATH = os.path.join(MODEL_DIR, "scaler.pkl")
DBSCAN_MODEL_PATH = os.path.join(MODEL_DIR, "dbscan_model.pkl")
IF_MODEL_PATH = os.path.join(MODEL_DIR, "if_model.pkl")

## Carga datos

In [6]:
print("Cargando y preprocesando datos...")
try:
    df_transactions = pl.read_parquet(FILE_PATH)
    df_transactions = df_transactions.with_columns([
        pl.col("transaction_amount").cast(pl.Float64),
        pl.col("transaction_date").cast(pl.Datetime)
    ]).unique() # Eliminar duplicados
    print(f"Datos cargados. Dimensiones: {df_transactions.shape}")
except Exception as e:
    print(f"Error cargando o preprocesando el archivo: {e}")

Cargando y preprocesando datos...
Datos cargados. Dimensiones: (10758414, 8)


In [7]:
# Extraer componentes de fecha para el análisis temporal 
df_transactions = df_transactions.with_columns([
    pl.col("transaction_date").dt.hour().alias("hour"),
    pl.col("transaction_date").dt.weekday().alias("day_of_week"),
    pl.col("transaction_date").dt.month().alias("month"),
    pl.col("transaction_date").dt.date().alias("transaction_day")
])

## Filtrado por heurísticas y Feature Engineering

In [8]:
print("Generando grupos candidatos y features avanzadas...")
# Definir la ventana de tiempo para el fraccionamiento
WINDOW_DURATION = "24h" 
MIN_TRANSACTIONS_IN_GROUP = 3 # Un umbral inicial de al menos 3 transacciones

candidate_groups_df = df_transactions.sort("transaction_date").group_by_dynamic(  # Agregar sort
    index_column="transaction_date",
    every=WINDOW_DURATION,
    group_by=["user_id", "merchant_id", "transaction_type"] 
).agg([
    pl.len().alias("transaction_count"), 
    pl.col("transaction_amount").sum().alias("total_group_amount"),
    pl.col("transaction_amount").std().alias("amount_stdev"),
    pl.col("transaction_amount").median().alias("amount_median"),
    pl.col("transaction_amount").min().alias("amount_min"),
    pl.col("transaction_amount").max().alias("amount_max"),
    (pl.col("transaction_date").max() - pl.col("transaction_date").min()).dt.total_minutes().alias("time_span_minutes"),
    pl.col("transaction_amount").implode().alias("all_amounts_in_group"),
    pl.col("transaction_date").implode().alias("all_dates_in_group") 
]).filter(
    pl.col("transaction_count") >= MIN_TRANSACTIONS_IN_GROUP
).with_columns([
    # Coeficiente de Variación: std / mean. Cuidado con mean=0
    (pl.col("amount_stdev") / pl.col("total_group_amount") * pl.col("transaction_count")).alias("amount_coeff_of_variation").fill_nan(0.0),
    # Entropía de los Montos
    pl.col("all_amounts_in_group").map_elements(calculate_amount_entropy, return_dtype=pl.Float64).alias("amount_entropy"), 
    # Calcular promedio de delta de tiempo entre transacciones dentro del grupo
    pl.col("all_dates_in_group").map_elements( 
        lambda dates: np.mean(np.diff(np.sort([d.timestamp() for d in dates.to_list()]))) / 60 if len(dates.to_list()) > 1 else 0.0,
        return_dtype=pl.Float64
    ).alias("avg_time_delta_minutes"),
    # Flag para montos redondos (ej. múltiplos de 100000)
    pl.col("all_amounts_in_group").map_elements(
        lambda amounts: any(a % 100000 == 0 for a in amounts.to_list()),
        return_dtype=pl.Boolean
    ).alias("has_round_amounts")
])


Generando grupos candidatos y features avanzadas...


In [9]:
# Limpiar columnas auxiliares
candidate_groups_df = candidate_groups_df.drop(["all_amounts_in_group", "all_dates_in_group"])

print(f"Total de grupos candidatos para ML: {candidate_groups_df.shape[0]}")
print(candidate_groups_df.head())

Total de grupos candidatos para ML: 175068
shape: (5, 15)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ user_id   ┆ merchant_ ┆ transacti ┆ transacti ┆ … ┆ amount_co ┆ amount_en ┆ avg_time_ ┆ has_roun │
│ ---       ┆ id        ┆ on_type   ┆ on_date   ┆   ┆ eff_of_va ┆ tropy     ┆ delta_min ┆ d_amount │
│ str       ┆ ---       ┆ ---       ┆ ---       ┆   ┆ riation   ┆ ---       ┆ utes      ┆ s        │
│           ┆ str       ┆ str       ┆ datetime[ ┆   ┆ ---       ┆ f64       ┆ ---       ┆ ---      │
│           ┆           ┆           ┆ μs]       ┆   ┆ f64       ┆           ┆ f64       ┆ bool     │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 0000ab49a ┆ 817d18cd3 ┆ DEBITO    ┆ 2021-10-2 ┆ … ┆ 0.908625  ┆ 1.098612  ┆ 669.725   ┆ false    │
│ f4f93dbd9 ┆ c31e40e9b ┆           ┆ 9         ┆   ┆           ┆           ┆           ┆          │
│ 9c2b3f2e5 ┆ ff0566baa ┆        

## Normalización y Encoding

In [18]:
print("Preparando features para los modelos ML...")
# Definir las features numéricas que serán usadas por IF y DBSCAN
# Excluir IDs o features ya agregadas que no son numéricas para el modelo
numerical_features = [
    "transaction_count",
    "total_group_amount",
    "amount_stdev",
    "amount_median",
    "amount_min",
    "amount_max",
    "time_span_minutes",
    "amount_coeff_of_variation",
    "amount_entropy",
    "avg_time_delta_minutes"
]

# Asegurarse de que todas las columnas existan antes de seleccionar
numerical_features_exist = [col for col in numerical_features if col in candidate_groups_df.columns]
if len(numerical_features_exist) != len(numerical_features):
    print(f"Advertencia: Algunas features numéricas esperadas no existen. Usando: {numerical_features_exist}")
numerical_features = numerical_features_exist

X_candidates_polars = candidate_groups_df.select(numerical_features)

# Convertir a NumPy para Scikit-learn
X_candidates = X_candidates_polars.to_numpy()

# Normalización con StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_candidates)
print(f"Features escaladas. Dimensiones de X_scaled: {X_scaled.shape}")

try:
    # Guardar el scaler en disco
    with open(SCALER_PATH, "wb") as f:
        pkl.dump(scaler, f)
    print("Scaler guardado en disco.")
except Exception as e:
    print(f"Error al guardar el scaler: {e}")

Preparando features para los modelos ML...
Features escaladas. Dimensiones de X_scaled: (175068, 10)
Scaler guardado en disco.


## Modelos

In [23]:
# Configuración de MLFlow
mlflow.set_experiment("transaction-splitting-detection")

# Parámetros para iterar
if_params_list = [
    {"n_estimators": 100, "contamination": 'auto', "random_state": 42},
    {"n_estimators": 200, "contamination": 0.01, "random_state": 42},
]

dbscan_params_list = [
    {"eps": 0.5, "min_samples": 5},
    {"eps": 0.3, "min_samples": 10},
]

# Iterar sobre las combinaciones de parámetros para encontrar el mejor modelo
run_id_counter = 0
for if_params in if_params_list:
    for dbscan_params in dbscan_params_list:
        run_id_counter += 1
        with mlflow.start_run(run_name=f"Run_{run_id_counter}"):
            print(f"\n--- Ejecutando MLFlow Run {run_id_counter} ---")
            print(f"IF Params: {if_params}")
            print(f"DBSCAN Params: {dbscan_params}")

            # Logear parámetros en MLFlow
            mlflow.log_params({"if_" + k: v for k, v in if_params.items()})
            mlflow.log_params({"dbscan_" + k: v for k, v in dbscan_params.items()})

            # --- Entrenamiento de Isolation Forest ---
            iso_forest = IsolationForest(**if_params)
            iso_forest.fit(X_scaled)
            # anomaly_scores_if: Menor valor = Más anómalo
            anomaly_scores_if = iso_forest.decision_function(X_scaled)
            # Convertir a un score de riesgo: Mayor valor = Más riesgo (normalizar 0-1)
            min_score, max_score = anomaly_scores_if.min(), anomaly_scores_if.max()
            risk_scores_if = 1 - (anomaly_scores_if - min_score) / (max_score - min_score)
            
            # --- Entrenamiento de DBSCAN ---
            dbscan = DBSCAN(**dbscan_params)
            # labels: -1 para ruido (anomalías), 0, 1, ... para clústeres
            dbscan_labels = dbscan.fit_predict(X_scaled)
            # is_noise: 1 si es ruido, 0 si es parte de un clúster
            is_dbscan_noise = (dbscan_labels == -1).astype(int)

            # --- Ensamble de Scores ---
            # Score combinado: 
            # Aquí se usa una combinación simple de los scores normalizados
            # Ponderación 0.7 para IF, 0.3 para DBSCAN (ajustable)
            combined_score = 0.7 * risk_scores_if + 0.3 * is_dbscan_noise

            # Añadir scores al DataFrame de candidatos (para análisis posterior)
            candidate_groups_with_scores = candidate_groups_df.with_columns([
                pl.Series("isolation_forest_score", anomaly_scores_if),
                pl.Series("dbscan_label", dbscan_labels),
                pl.Series("is_dbscan_noise", is_dbscan_noise),
                pl.Series("combined_risk_score", combined_score)
            ])

            # --- Evaluación y Registro de Métricas ---
            # Para modelos no supervisados sin ground truth, la evaluación es desafiante.
            # Aquí se usan métricas intrínsecas y un umbral para simular Precision/Recall.

            # Métrica intrínseca de clustering (para DBSCAN)
            # silhouette_score requiere al menos 2 clústeres y 2 muestras.
            try:
                if len(np.unique(dbscan_labels)) > 1 and len(dbscan_labels) > 1:
                    silhouette_avg = silhouette_score(X_scaled, dbscan_labels)
                    mlflow.log_metric("dbscan_silhouette_score", silhouette_avg)
                    print(f"DBSCAN Silhouette Score: {silhouette_avg:.4f}")
                else:
                    print("No suficientes clústeres para calcular Silhouette Score.")
            except Exception as e:
                print(f"Error calculando Silhouette Score: {e}")

            # Umbral de riesgo combinado (para simular alertas)
            # Por ejemplo, top 5% de los scores más altos son alertas
            alert_threshold = np.percentile(combined_score, 95) # Top 5% más riesgoso
            num_alerts = (combined_score >= alert_threshold).sum()
            
            mlflow.log_metric("alert_threshold", alert_threshold)
            mlflow.log_metric("num_alerts", num_alerts)
            print(f"Alertas generadas (top 5%): {num_alerts}")

            # Evaluación de la "Calidad" de las Alertas sin etiquetas reales:
            # Se puede examinar las características de los grupos de "alto riesgo"
            high_risk_groups = candidate_groups_with_scores.filter(
                pl.col("combined_risk_score") >= alert_threshold
            )
            
            # Se puede calcular la "uniformidad promedio" de los montos en los grupos de alto riesgo
            avg_entropy_high_risk = high_risk_groups.select("amount_entropy").mean().item()
            avg_coeff_var_high_risk = high_risk_groups.select("amount_coeff_of_variation").mean().item()

            mlflow.log_metric("avg_entropy_high_risk", avg_entropy_high_risk)
            mlflow.log_metric("avg_coeff_var_high_risk", avg_coeff_var_high_risk)
            print(f"Entropía promedio en alertas: {avg_entropy_high_risk:.4f}")
            print(f"Coeff. Var. promedio en alertas: {avg_coeff_var_high_risk:.4f}")
            # Se espera que estos valores sean BAJOS para alertas de fraccionamiento

            # --- Registro de Modelos y Artefactos en MLFlow ---
            mlflow.sklearn.log_model(iso_forest, "isolation_forest_model")
            mlflow.sklearn.log_model(dbscan, "dbscan_model")
            mlflow.sklearn.log_model(scaler, "feature_scaler") # Es crucial guardar el scaler

            # Guardar el DataFrame de grupos con scores como artefacto
            # Guardar solo top N para evitar archivos muy grandes
            output_df_for_logging = candidate_groups_with_scores.sort("combined_risk_score", descending=True).head(5000) # Top 5000
            output_df_for_logging.write_parquet("high_risk_candidate_groups.parquet")
            mlflow.log_artifact("high_risk_candidate_groups.parquet")

        	# Guardar un gráfico de dispersión de los scores
            plot_data = candidate_groups_with_scores.select([
                "amount_entropy", "amount_coeff_of_variation", "combined_risk_score"
            ]).sample(n=min(50000, candidate_groups_with_scores.height)).to_pandas() # Muestra para plotear

            plt.figure(figsize=(10, 8))
            scatter = plt.scatter(x=plot_data['amount_entropy'], 
                                y=plot_data['amount_coeff_of_variation'], 
                                c=plot_data['combined_risk_score'], 
                                cmap='viridis', 
                                alpha=0.6)
            plt.title('Ensemble Score vs. Entropía y Coeficiente de Variación')
            plt.xlabel('Entropía de los Montos')
            plt.ylabel('Coeficiente de Variación de los Montos')
            plt.colorbar(scatter, label='Combined Risk Score')  # Usar el objeto scatter
            plt.savefig("ensemble_score_scatter.png")
            mlflow.log_artifact("ensemble_score_scatter.png")
            plt.close()

            print(f"MLFlow Run {run_id_counter} completado. View run at: {mlflow.get_tracking_uri()}")

print("\n--- Proceso de Modelado Completado ---")
print("Puedes revisar los resultados en la UI de MLFlow ejecutando: 'mlflow ui' en tu terminal.")


--- Ejecutando MLFlow Run 1 ---
IF Params: {'n_estimators': 100, 'contamination': 'auto', 'random_state': 42}
DBSCAN Params: {'eps': 0.5, 'min_samples': 5}
DBSCAN Silhouette Score: -0.4152
Alertas generadas (top 5%): 8754


2025/06/26 11:53:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Entropía promedio en alertas: 0.9018
Coeff. Var. promedio en alertas: 0.5860


2025/06/26 11:53:48 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/06/26 11:53:48 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/06/26 11:53:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/06/26 11:53:48 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2025/06/26 11:53:50 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/06/26 11:53:50 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/06/26 11:53:50 WARNING ml

MLFlow Run 1 completado. View run at: file:///d:/PROYECTOS/Nequi/transaction-splitting/notebooks/mlruns

--- Ejecutando MLFlow Run 2 ---
IF Params: {'n_estimators': 100, 'contamination': 'auto', 'random_state': 42}
DBSCAN Params: {'eps': 0.3, 'min_samples': 10}


...

MLFlow Run 3 completado. View run at: file:///d:/PROYECTOS/Nequi/transaction-splitting/notebooks/mlruns

--- Ejecutando MLFlow Run 4 ---
IF Params: {'n_estimators': 200, 'contamination': 0.01, 'random_state': 42}
DBSCAN Params: {'eps': 0.3, 'min_samples': 10}
DBSCAN Silhouette Score: -0.4564
Alertas generadas (top 5%): 8754


2025/06/26 12:07:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Entropía promedio en alertas: 0.9842
Coeff. Var. promedio en alertas: 0.6394


2025/06/26 12:07:12 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/06/26 12:07:12 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/06/26 12:07:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/06/26 12:07:13 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2025/06/26 12:07:15 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/06/26 12:07:15 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/06/26 12:07:15 WARNING ml

MLFlow Run 4 completado. View run at: file:///d:/PROYECTOS/Nequi/transaction-splitting/notebooks/mlruns

--- Proceso de Modelado Completado ---
Puedes revisar los resultados en la UI de MLFlow ejecutando: 'mlflow ui' en tu terminal.


Se ha encontrado una buena combinación de parámetros inicial, pero hay margen significativo para mejorar el componente DBSCAN y afinar el ensamble, siempre buscando reducir aún más la entropía y el coeficiente de variación de los montos en los grupos alertados.

In [14]:
# Configuración de MLFlow
mlflow.set_experiment("transaction-splitting-detection_2")

# Parámetros para Isolation Forest
if_params_grid = {
    "n_estimators": [300],
    "contamination": [0.001, 0.005, 0.02],
    "random_state": [42] # Mantener fijo para reproducibilidad
}

# Parámetros para DBSCAN
dbscan_params_grid = {
    "eps": [0.1, 0.2, 0.9],
    "min_samples": [3, 15, 20]
}

# Generar todas las combinaciones de parámetros usando itertools.product
if_param_combinations = list(itertools.product(
    if_params_grid["n_estimators"],
    if_params_grid["contamination"],
    if_params_grid["random_state"]
))

dbscan_param_combinations = list(itertools.product(
    dbscan_params_grid["eps"],
    dbscan_params_grid["min_samples"]
))

# Convertir a listas de diccionarios para mayor claridad y facilidad de uso
if_params_list = []
for n_est, contam, r_state in if_param_combinations:
    if_params_list.append({
        "n_estimators": n_est,
        "contamination": contam,
        "random_state": r_state
    })

dbscan_params_list = []
for eps_val, min_samp_val in dbscan_param_combinations:
    dbscan_params_list.append({
        "eps": eps_val,
        "min_samples": min_samp_val
    })

# Iterar sobre las combinaciones de parámetros para encontrar el mejor modelo
run_id_counter = 0
for if_params in if_params_list:
    for dbscan_params in dbscan_params_list:
        run_id_counter += 1
        print(f"\n--- Ejecutando MLFlow Run {run_id_counter} ---")

        try:
            with mlflow.start_run() as run:
                # Log de parámetros de los modelos
                mlflow.log_params({"if_" + k: v for k, v in if_params.items()})
                mlflow.log_params({"dbscan_" + k: v for k, v in dbscan_params.items()})
                
                print(f"IF Params: {if_params}")
                print(f"DBSCAN Params: {dbscan_params}")

                # 1. Entrenamiento de Isolation Forest
                try:
                    iso_forest = IsolationForest(**if_params)
                    iso_forest.fit(X_scaled)
                    anomaly_scores_if = iso_forest.decision_function(X_scaled)
                except Exception as e:
                    print(f"Error en Isolation Forest: {e}")
                    mlflow.log_param("isolation_forest_error", str(e))
                    continue
                
                # 2. Entrenamiento de DBSCAN
                try:
                    dbscan = DBSCAN(**dbscan_params)
                    dbscan_labels = dbscan.fit_predict(X_scaled)
                    is_dbscan_noise = (dbscan_labels == -1).astype(int)
                except Exception as e:
                    print(f"Error en DBSCAN: {e}")
                    mlflow.log_param("dbscan_error", str(e))
                    continue

                # 3. Métricas para DBSCAN (Silhouette Score) - Lógica simplificada
                unique_labels = np.unique(dbscan_labels)
                non_noise_labels = unique_labels[unique_labels != -1]
                
                if len(non_noise_labels) >= 2:
                    non_noise_indices = dbscan_labels != -1
                    if np.sum(non_noise_indices) > 1:
                        try:
                            silhouette_avg = silhouette_score(X_scaled[non_noise_indices], dbscan_labels[non_noise_indices])
                            mlflow.log_metric("dbscan_silhouette_score_non_noise", silhouette_avg)
                            print(f"DBSCAN Silhouette Score (non-noise): {silhouette_avg:.4f}")
                        except Exception as e:
                            print(f"Error calculando Silhouette Score: {e}")
                            mlflow.log_metric("dbscan_silhouette_score_non_noise", -1.0)
                    else:
                        mlflow.log_metric("dbscan_silhouette_score_non_noise", -1.0)
                        print("DBSCAN Silhouette Score: No aplicable (pocos puntos no-ruido)")
                else:
                    mlflow.log_metric("dbscan_silhouette_score_non_noise", -1.0)
                    print("DBSCAN Silhouette Score: No aplicable (menos de 2 clusters)")

                # Log número de clusters y ruido
                mlflow.log_metric("num_clusters", len(non_noise_labels))
                mlflow.log_metric("num_noise_points", np.sum(is_dbscan_noise))

                # 4. Normalización y Combinación de Scores
                try:
                    # Escalar los scores de Isolation Forest a [0, 1] (0 = normal, 1 = anómalo)
                    min_score_if = np.min(anomaly_scores_if)
                    max_score_if = np.max(anomaly_scores_if)
                    
                    if max_score_if != min_score_if:
                        risk_scores_if = 1 - (anomaly_scores_if - min_score_if) / (max_score_if - min_score_if)
                    else:
                        risk_scores_if = np.zeros_like(anomaly_scores_if)
                    
                    # Combinar scores (ej. 70% IF, 30% DBSCAN como indicador de ruido)
                    combined_risk_score = 0.7 * risk_scores_if + 0.3 * is_dbscan_noise
                except Exception as e:
                    print(f"Error en combinación de scores: {e}")
                    continue

                # Añadir scores al DataFrame de grupos candidatos
                try:
                    candidate_groups_scores_df = candidate_groups_df.with_columns([
                        pl.Series("isolation_forest_score", anomaly_scores_if),
                        pl.Series("dbscan_label", dbscan_labels),
                        pl.Series("is_dbscan_noise", is_dbscan_noise),
                        pl.Series("combined_risk_score", combined_risk_score)
                    ])
                except Exception as e:
                    print(f"Error añadiendo scores al DataFrame: {e}")
                    continue

                # 5. Generación y Evaluación de Alertas (ej. top 5% de los scores más altos)
                try:
                    alert_threshold = np.percentile(combined_risk_score, 95) # Top 5%
                    high_risk_groups = candidate_groups_scores_df.filter(
                        pl.col("combined_risk_score") >= alert_threshold
                    )
                    
                    num_alerts = high_risk_groups.shape[0]
                    mlflow.log_metric("alerts_generated_top_5_percent", num_alerts)
                    mlflow.log_metric("alert_threshold", alert_threshold)
                    print(f"Alertas generadas (top 5%): {num_alerts}")
                except Exception as e:
                    print(f"Error generando alertas: {e}")
                    mlflow.log_metric("alerts_generated_top_5_percent", -1)

                # 6. Evaluación de la calidad de las alertas por entropía y coeficiente de variación
                try:
                    if num_alerts > 0:
                        avg_entropy_high_risk = high_risk_groups.select("amount_entropy").mean().item()
                        avg_coeff_var_high_risk = high_risk_groups.select("amount_coeff_of_variation").mean().item()
                        
                        mlflow.log_metric("entropia_promedio_en_alertas", avg_entropy_high_risk)
                        mlflow.log_metric("coeff_var_promedio_en_alertas", avg_coeff_var_high_risk)
                        
                        print(f"Entropía promedio en alertas: {avg_entropy_high_risk:.4f}")
                        print(f"Coeff. Var. promedio en alertas: {avg_coeff_var_high_risk:.4f}")
                    else:
                        mlflow.log_metric("entropia_promedio_en_alertas", -1.0)
                        mlflow.log_metric("coeff_var_promedio_en_alertas", -1.0)
                        print("No se generaron alertas para calcular métricas de calidad.")
                except Exception as e:
                    print(f"Error evaluando calidad de alertas: {e}")

                # 7. Loggeo de los Modelos y el Scaler
                try:
                    # Input example para Isolation Forest y Scaler
                    input_example = X_scaled[:1] if X_scaled.shape[0] > 0 else None

                    mlflow.sklearn.log_model(
                        sk_model=iso_forest,
                        artifact_path="isolation_forest_model",
                        input_example=input_example
                    )
                    
                    # DBSCAN sin input_example para evitar problemas
                    mlflow.sklearn.log_model(
                        sk_model=dbscan,
                        artifact_path="dbscan_model"
                    )
                    
                    # Input example para el scaler usando datos originales
                    scaler_input = X_candidates_polars.head(1).to_numpy() if X_candidates_polars.shape[0] > 0 else None
                    mlflow.sklearn.log_model(
                        sk_model=scaler,
                        artifact_path="feature_scaler",
                        input_example=scaler_input
                    )
                except Exception as e:
                    print(f"Error guardando modelos: {e}")
                
                # 8. Guardar gráfico de dispersión de los scores
                try:
                    if candidate_groups_scores_df.height > 0:
                        plot_data = candidate_groups_scores_df.select([
                            "amount_entropy", "amount_coeff_of_variation", "combined_risk_score"
                        ]).sample(n=min(50000, candidate_groups_scores_df.height)).to_pandas()

                        plt.figure(figsize=(10, 8))
                        scatter = plt.scatter(x=plot_data['amount_entropy'], 
                                            y=plot_data['amount_coeff_of_variation'], 
                                            c=plot_data['combined_risk_score'], 
                                            cmap='viridis', 
                                            alpha=0.6)
                        plt.title('Ensemble Score vs. Entropía y Coeficiente de Variación')
                        plt.xlabel('Entropía de los Montos')
                        plt.ylabel('Coeficiente de Variación de los Montos')
                        plt.colorbar(scatter, label='Combined Risk Score')
                        plt.savefig("ensemble_score_scatter.png", dpi=150, bbox_inches='tight')
                        mlflow.log_artifact("ensemble_score_scatter.png")
                        plt.close()
                    else:
                        print("No hay datos para plotear")
                except Exception as e:
                    print(f"Error creando gráfico: {e}")
                
                print(f"MLFlow Run {run_id_counter} completado. View run at: {mlflow.get_tracking_uri()}")

        except Exception as e:
            print(f"Error general en Run {run_id_counter}: {e}")
            # Log del error en MLflow si es posible
            try:
                mlflow.log_param("general_error", str(e))
            except:
                pass
            continue

print("\n--- Proceso de Modelado Completado ---")
print("Puedes revisar los resultados en la UI de MLFlow ejecutando: 'mlflow ui' en tu terminal.")


--- Ejecutando MLFlow Run 1 ---
IF Params: {'n_estimators': 300, 'contamination': 0.001, 'random_state': 42}
DBSCAN Params: {'eps': 0.1, 'min_samples': 3}
DBSCAN Silhouette Score (non-noise): -0.3056


2025/06/26 14:26:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Alertas generadas (top 5%): 8754
Entropía promedio en alertas: 0.9794
Coeff. Var. promedio en alertas: 0.6474


2025/06/26 14:26:33 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/06/26 14:26:34 INFO mlflow.models.model: Found the following environment variables used during model inference: [GEMINI_API_KEY, HF_API_KEY, HUGGINGFACEHUB_API_TOKEN, ... ]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
2025/06/26 14:26:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/06/26 14:26:34 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2025/06/26 14:26:37 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/06/26 14:26:37 WARNING mlflow.models.model: Model logged without a sign

MLFlow Run 1 completado. View run at: file:///d:/PROYECTOS/Nequi/transaction-splitting/notebooks/mlruns

--- Ejecutando MLFlow Run 2 ---
IF Params: {'n_estimators': 300, 'contamination': 0.001, 'random_state': 42}
DBSCAN Params: {'eps': 0.1, 'min_samples': 15}


...

2025/06/26 15:45:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Alertas generadas (top 5%): 8760
Entropía promedio en alertas: 0.8385
Coeff. Var. promedio en alertas: 0.5791


2025/06/26 15:45:06 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/06/26 15:45:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/06/26 15:45:06 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2025/06/26 15:45:08 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/06/26 15:45:08 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/06/26 15:45:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/06/26 15:45:08 WARNING mlflow.models.signature: Failed to infer the model signature from the input example. Reason

MLFlow Run 27 completado. View run at: file:///d:/PROYECTOS/Nequi/transaction-splitting/notebooks/mlruns

--- Ejecutando MLFlow Run 28 ---
IF Params: {'n_estimators': 500, 'contamination': 0.001, 'random_state': 42}
DBSCAN Params: {'eps': 0.1, 'min_samples': 3}


KeyboardInterrupt: 

Guardar en loca el mejor modelo


In [20]:
# Entrenar y guardar en local la mejor combinación de IS y DBSCAN
dbscan_eps=0.9
dbscan_min_samples=15
if_contamination=0.02
if_n_estimators=300


# 1. Entrenamiento de Isolation Forest
if_params = {
    "contamination": if_contamination,
    "n_estimators": if_n_estimators,
    "random_state": 42
}
iso_forest = IsolationForest(**if_params)
iso_forest.fit(X_scaled)
anomaly_scores_if = iso_forest.decision_function(X_scaled)


# 2. Entrenamiento de DBSCAN
dbscan_params = {
    "eps": dbscan_eps,
    "min_samples": dbscan_min_samples
}
dbscan = DBSCAN(**dbscan_params)
dbscan_labels = dbscan.fit_predict(X_scaled)
is_dbscan_noise = (dbscan_labels == -1).astype(int)

# 3. Guardar los modelos en disco
with open(IF_MODEL_PATH, "wb") as f:
    pkl.dump(iso_forest, f)

with open(DBSCAN_MODEL_PATH, "wb") as f:
    pkl.dump(dbscan, f)

## Inferencia

In [11]:
import polars as pl
import numpy as np
import mlflow
import mlflow.sklearn

In [12]:
BEST_RUN_ID = "3bcc0b92f8ca4f31862e9e40d3578359" # Ejemplo: "a1b2c3d4e5f6g7h8i9j0k1l2m3n4o5p6"

In [13]:
print(f"Cargando el modelo y el scaler del Run ID: {BEST_RUN_ID}")
try:
    # Cargar el scaler de features
    scaler = mlflow.sklearn.load_model(f"runs:/{BEST_RUN_ID}/feature_scaler")
    
    # Cargar el modelo Isolation Forest
    iso_forest = mlflow.sklearn.load_model(f"runs:/{BEST_RUN_ID}/isolation_forest_model")
    
    # Cargar el modelo DBSCAN
    dbscan = mlflow.sklearn.load_model(f"runs:/{BEST_RUN_ID}/dbscan_model")
    
    
    print("Modelos y scaler cargados exitosamente.")
except Exception as e:
    print(f"Error cargando los modelos o el scaler desde MLFlow: {e}")
    print("Asegúrate de que el RUN_ID sea correcto y que los artefactos se hayan loggeado.")
    exit()

Cargando el modelo y el scaler del Run ID: 3bcc0b92f8ca4f31862e9e40d3578359
Modelos y scaler cargados exitosamente.


In [14]:
def predict_fraccionamiento(new_transactions_df: pl.DataFrame) -> pl.DataFrame:
    """
    Realiza la inferencia de fraccionamiento en un nuevo DataFrame de transacciones.
    
    Args:
        new_transactions_df: DataFrame de Polars con las nuevas transacciones,
                            con las mismas columnas que el DataFrame original.
                            (user_id, merchant_id, transaction_date, transaction_amount, transaction_type)
    
    Returns:
        DataFrame de Polars con los grupos candidatos y sus scores de riesgo,
        incluyendo una columna 'is_fraccionamiento_alert'.
    """
    print("Iniciando preprocesamiento de nuevas transacciones...")

    # A. Preprocesamiento Básico para Nuevos Datos (igual que en entrenamiento)
    try:
        # Primero intentar parsear las fechas si están como string
        new_transactions_df = new_transactions_df.with_columns([
            pl.col("transaction_amount").cast(pl.Float64),
            # Parsear fechas desde string con formato específico
            pl.col("transaction_date").str.to_datetime("%Y-%m-%d %H:%M:%S").alias("transaction_date")
        ]).unique() # Eliminar duplicados si los hay
    except Exception as e:
        print(f"Error en conversión de fechas, intentando conversión directa: {e}")
        try:
            # Si falla, intentar conversión directa
            new_transactions_df = new_transactions_df.with_columns([
                pl.col("transaction_amount").cast(pl.Float64),
                pl.col("transaction_date").cast(pl.Datetime)
            ]).unique()
        except Exception as e2:
            print(f"Error en conversión directa: {e2}")
            # Último intento con strict=False
            new_transactions_df = new_transactions_df.with_columns([
                pl.col("transaction_amount").cast(pl.Float64),
                pl.col("transaction_date").str.to_datetime(strict=False).alias("transaction_date")
            ]).unique()
    
    # Extraer componentes de fecha
    new_transactions_df = new_transactions_df.with_columns([
        pl.col("transaction_date").dt.hour().alias("hour"),
        pl.col("transaction_date").dt.weekday().alias("day_of_week"),
        pl.col("transaction_date").dt.month().alias("month"),
        pl.col("transaction_date").dt.date().alias("transaction_day")
    ])

    # B. Filtrado Heurístico y Feature Engineering Avanzado para Nuevos Datos
    # Las constantes deben ser las mismas que en entrenamiento
    WINDOW_DURATION = "24h"  
    MIN_TRANSACTIONS_IN_GROUP = 3

    print("Generando grupos candidatos y features para nuevas transacciones...")
    new_candidate_groups_df = new_transactions_df.sort("transaction_date").group_by_dynamic(  # Agregar sort
        index_column="transaction_date",
        every=WINDOW_DURATION,
        group_by=["user_id", "merchant_id", "transaction_type"] 
    ).agg([
        pl.len().alias("transaction_count"), 
        pl.col("transaction_amount").sum().alias("total_group_amount"),
        pl.col("transaction_amount").std().alias("amount_stdev"),
        pl.col("transaction_amount").median().alias("amount_median"),
        pl.col("transaction_amount").min().alias("amount_min"),
        pl.col("transaction_amount").max().alias("amount_max"),
        (pl.col("transaction_date").max() - pl.col("transaction_date").min()).dt.total_minutes().alias("time_span_minutes"),
        pl.col("transaction_amount").implode().alias("all_amounts_in_group"), 
        pl.col("transaction_date").implode().alias("all_dates_in_group") 
    ]).filter(
        pl.col("transaction_count") >= MIN_TRANSACTIONS_IN_GROUP
    ).with_columns([
        (pl.col("amount_stdev") / pl.col("total_group_amount") * pl.col("transaction_count")).alias("amount_coeff_of_variation").fill_nan(0.0),
        pl.col("all_amounts_in_group").map_elements(calculate_amount_entropy, return_dtype=pl.Float64).alias("amount_entropy"),
        pl.col("all_dates_in_group").map_elements(
            lambda dates: np.mean(np.diff(np.sort([d.timestamp() for d in dates.to_list()]))) / 60 if len(dates.to_list()) > 1 else 0.0,
            return_dtype=pl.Float64
        ).alias("avg_time_delta_minutes"),
        pl.col("all_amounts_in_group").map_elements(
            lambda amounts: any(a % 100000 == 0 for a in amounts.to_list()),
            return_dtype=pl.Boolean
        ).alias("has_round_amounts")
    ]).drop(["all_amounts_in_group", "all_dates_in_group"])

    print(f"Total de nuevos grupos candidatos para inferencia: {new_candidate_groups_df.shape[0]}")
    
    if new_candidate_groups_df.shape[0] == 0:
        print("No se encontraron nuevos grupos candidatos. Retornando DataFrame vacío.")
        # Crear schema básico para retorno
        empty_schema = {
            "user_id": pl.Int64,
            "merchant_id": pl.Int64,
            "transaction_type": pl.Utf8,
            "transaction_count": pl.Int64,
            "combined_risk_score": pl.Float64,
            "is_fraccionamiento_alert": pl.Boolean
        }
        return pl.DataFrame(schema=empty_schema)

    # Definir las features numéricas que serán usadas por IF y DBSCAN
    numerical_features = [
        "transaction_count", "total_group_amount", "amount_stdev", "amount_median",
        "amount_min", "amount_max", "time_span_minutes",
        "amount_coeff_of_variation", "amount_entropy", "avg_time_delta_minutes"
    ]
    
    # Asegurarse de que todas las columnas existan antes de seleccionar
    numerical_features_exist = [col for col in numerical_features if col in new_candidate_groups_df.columns]
    if len(numerical_features_exist) != len(numerical_features):
        print(f"Advertencia en inferencia: Algunas features numéricas esperadas no existen. Usando: {numerical_features_exist}")
    numerical_features = numerical_features_exist

    X_new_candidates_polars = new_candidate_groups_df.select(numerical_features)
    X_new_candidates = X_new_candidates_polars.to_numpy()

    # C. Escalar Nuevos Datos 
    X_new_scaled = scaler.transform(X_new_candidates)
    print("Nuevas features escaladas.")

    # D. Generar Scores con los Modelos Cargados
    # Isolation Forest
    anomaly_scores_if_new = iso_forest.decision_function(X_new_scaled)
    
    # Normalizar scores de Isolation Forest a [0, 1] basado en los scores actuales
    min_score_if = anomaly_scores_if_new.min()
    max_score_if = anomaly_scores_if_new.max()
    
    if max_score_if != min_score_if:
        risk_scores_if_new = 1 - (anomaly_scores_if_new - min_score_if) / (max_score_if - min_score_if)
    else:
        risk_scores_if_new = np.zeros_like(anomaly_scores_if_new)

    # DBSCAN - usar fit_predict para nuevos datos
    dbscan_labels_new = dbscan.fit_predict(X_new_scaled)
    is_dbscan_noise_new = (dbscan_labels_new == -1).astype(int)

    # E. Combinar Scores (usando las mismas ponderaciones)
    combined_score_new = 0.7 * risk_scores_if_new + 0.3 * is_dbscan_noise_new
    print("Scores de riesgo calculados.")

    # F. Añadir Scores al DataFrame de nuevos candidatos
    new_candidate_groups_with_scores = new_candidate_groups_df.with_columns([
        pl.Series("isolation_forest_score", anomaly_scores_if_new),
        pl.Series("dbscan_label", dbscan_labels_new),
        pl.Series("is_dbscan_noise", is_dbscan_noise_new),
        pl.Series("combined_risk_score", combined_score_new)
    ])

    # G. Generar Alertas (top 5% de riesgo)
    alert_threshold = np.percentile(combined_score_new, 95) 
    
    final_alerts_df = new_candidate_groups_with_scores.with_columns(
        (pl.col("combined_risk_score") >= alert_threshold).alias("is_fraccionamiento_alert")
    )
    
    num_alerts = final_alerts_df.filter(pl.col("is_fraccionamiento_alert")).shape[0]
    print(f"Total de alertas de fraccionamiento generadas: {num_alerts}")

    return final_alerts_df

In [10]:

dummy_new_data = pl.DataFrame({
    "_id": ["new_tx_1", "new_tx_2", "new_tx_3", "new_tx_4", "new_tx_5", "new_tx_6"],
    "merchant_id": ["MERCH001", "MERCH001", "MERCH001", "MERCH001", "MERCH002", "MERCH003"],
    "subsidiary": ["SUB001", "SUB001", "SUB001", "SUB001", "SUB002", "SUB003"],
    "transaction_date": ["2025-06-26 10:00:00", "2025-06-25 10:02:00", "2025-06-25 10:04:00", "2025-06-25 10:06:00", "2025-06-25 10:08:00", "2025-06-25 12:01:00"],
    "account_number": ["ACC001", "ACC001", "ACC001", "ACC002", "ACC003", "ACC003"],
    "user_id": ["USER001", "USER001", "USER001", "USER001", "USER001", "USER001"],
    "transaction_amount": [500000.0, 500000.0, 500000.0, 100000.0, 250000.0, 250000.0],
    "transaction_type": ["DEBIT", "DEBIT", "DEBIT", "DEBIT", "DEBIT", "CREDIT"]
})

In [15]:
inferred_results = predict_fraccionamiento(dummy_new_data)
print("\n--- Resultados de Inferencia para Nuevas Transacciones ---")
print(inferred_results)
print("\nGrupos marcados como alerta:")
print(inferred_results.filter(pl.col("is_fraccionamiento_alert")))

Iniciando preprocesamiento de nuevas transacciones...
Generando grupos candidatos y features para nuevas transacciones...
Total de nuevos grupos candidatos para inferencia: 1
Nuevas features escaladas.
Scores de riesgo calculados.
Total de alertas de fraccionamiento generadas: 1

--- Resultados de Inferencia para Nuevas Transacciones ---
shape: (1, 20)
┌─────────┬────────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ user_id ┆ merchant_i ┆ transacti ┆ transacti ┆ … ┆ dbscan_la ┆ is_dbscan ┆ combined_ ┆ is_fracci │
│ ---     ┆ d          ┆ on_type   ┆ on_date   ┆   ┆ bel       ┆ _noise    ┆ risk_scor ┆ onamiento │
│ str     ┆ ---        ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ e         ┆ _alert    │
│         ┆ str        ┆ str       ┆ datetime[ ┆   ┆ i64       ┆ i64       ┆ ---       ┆ ---       │
│         ┆            ┆           ┆ μs]       ┆   ┆           ┆           ┆ f64       ┆ bool      │
╞═════════╪════════════╪═══════════╪═══